# Chain of Thought on bAbI with `TransformerVarieties`

Chain of thought (CoT) is the trick of making a model **write its intermediate work into
the context** before committing to an answer. A decoder-only transformer does a fixed
amount of computation per token, so a question needing several sequential lookups cannot
be resolved in the single forward pass that produces the answer token. Letting the model
emit the intermediate facts first turns depth-in-layers into depth-in-tokens: each
retrieved fact is re-read through the whole stack of blocks, and the context becomes
working memory.

**Why bAbI.** Facebook's bAbI tasks are the rare dataset that ships with the chain of
thought already annotated. Every question comes with the line numbers of the *supporting
facts* needed to answer it, so we do not have to invent a rationale format — the dataset
tells us which sentences constitute the reasoning, and in what order:

```
1 Mice are afraid of wolves.
2 Gertrude is a mouse.
...
9 What is gertrude afraid of?      wolf      2 1
                                   ^answer   ^supporting facts, in reasoning order
```

That gives us two ways to render exactly the same question:

| Format | Target text |
| --- | --- |
| Direct | `... what is gertrude afraid of ? = # wolf` |
| Chain of thought | `... what is gertrude afraid of ? = gertrude is a mouse . mice are afraid of wolves . # wolf` |

It also gives us something the usual synthetic CoT demo cannot: a **gold standard for the
chain itself**. We can measure not just whether the answer is right, but whether the model
retrieved the sentences the dataset says are needed — a faithfulness metric that is not
our own invention.

We train on four tasks at once, chosen so the number of reasoning hops varies while the
vocabulary and surface form stay constant:

| Task | Name | Supporting facts |
| --- | --- | --- |
| 1 | single supporting fact | 1 |
| 15 | basic deduction | 2 |
| 16 | basic induction | 3 |
| 19 | path finding | 2 |

Task 1 is the **control**: a one-hop lookup needs no chain, so if CoT is doing what we
claim, its advantage should be near zero there and large on the multi-hop tasks. Same
model, same corpus, same optimizer budget — the only difference is whether the supporting
facts appear in the target.

Everything else is this repo: `TransformerMain.Transformer`, GQA + RoPE, the `Configs`
dataclass, and `DataLoader.tinyDataLoader` for batching.

In [1]:
import os, re, sys, time, random, tarfile, urllib.request, collections
import numpy as np
import torch
from torch.nn import functional as F

# run from the repo root so module imports and SavedModels/ paths resolve
sys.path.insert(0, os.getcwd())
import TransformerMain, Training, DataLoader, Generate

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)

torch 2.9.1+cu128 | device: cuda


---
## 1. Why not just prompt a saved checkpoint?

`loadModel` rebuilds a model from the config embedded in the checkpoint. The older
checkpoints here were saved before `use_experts` was added to `Configs`, so their stored
config has no such key and the dataclass default (`True`) makes `loadModel` build
mixture-of-experts blocks that the saved weights cannot fill. The helper below supplies
the key before constructing the model.

In [2]:
def load_checkpoint(name):
    '''Like TransformerMain.loadModel, but tolerant of checkpoints saved before
    `use_experts` existed in Configs (their config dict lacks the key, and the
    dataclass default of True would build MoE blocks the weights can't fill).'''
    ckpt = torch.load(os.path.join('SavedModels', name), weights_only=False)
    cfg = Training.Configs(**{'use_experts': False, **ckpt['config']})
    cfg.device = DEVICE
    model = TransformerMain.Transformer(cfg).to(DEVICE)
    model.load_state_dict(ckpt['state_dict'])
    return model.eval()

shakespeare = load_checkpoint('model_gqa_RoPE.pth')

# The TinyShakespeare character vocabulary, hardcoded so this cell needs no network
# (it is exactly DataLoader.getRawData()[1]).
SHAKESPEARE_VOCAB = "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
shakespeare_map = {k: v for v, k in enumerate(SHAKESPEARE_VOCAB)}

prompt = 'Mary went to the kitchen. Where is Mary?'
shakespeare.config.inference = False
print(Generate.generateGreedy(shakespeare, prompt, shakespeare_map,
                              shakespeare.config, max_gen=len(prompt) + 180))

Mary went to the kitchen. Where is Mary?

PETRUCHIO:
What is the sea the son the sea with the sea,
And then the sea the strange the sea,
And then the sea the strange the state of the seass,
And then the sea the strange t


It continues in Shakespeare and ignores the question. Nothing is wrong with the
checkpoint — a model can only produce a chain of thought if chains of thought were in its
training distribution. So the notebook keeps the architecture and swaps the data.

---
## 2. Getting bAbI

The 20 tasks, 10k questions each. The original Facebook URL is dead; the S3 mirror used
by the Keras example is still up. About 12 MB, extracted into `data/`.

In [3]:
BABI_URL = 'https://s3.amazonaws.com/text-datasets/babi_tasks_1-20_v1-2.tar.gz'
BABI_DIR = os.path.join('data', 'tasks_1-20_v1-2', 'en-10k')

if not os.path.isdir(BABI_DIR):
    os.makedirs('data', exist_ok=True)
    archive = os.path.join('data', 'babi.tar.gz')
    print('downloading bAbI ...')
    urllib.request.urlretrieve(BABI_URL, archive)
    with tarfile.open(archive) as tar:
        tar.extractall('data')
    print('extracted to data/')
print(sorted(os.listdir(BABI_DIR))[:4], '...')

['qa10_indefinite-knowledge_test.txt', 'qa10_indefinite-knowledge_train.txt', 'qa11_basic-coreference_test.txt', 'qa11_basic-coreference_train.txt'] ...


### Parsing

Each file is a stream of numbered lines. A line numbered `1` starts a new story; lines
containing a tab are questions, carrying `question \t answer \t supporting line numbers`.
A question is answered from the story *so far*, so we snapshot the accumulated facts at
each question.

In [4]:
def parse_babi(path):
    '''-> [(story {line_no: sentence}, question, answer, [supporting line numbers])]'''
    story, out = {}, []
    for line in open(path):
        num, rest = line.rstrip('\n').split(' ', 1)
        num = int(num)
        if num == 1:                       # line 1 starts a fresh story
            story = {}
        if '\t' in rest:                   # a question
            q, answer, support = rest.split('\t')
            out.append((dict(story), q.strip(), answer.strip(),
                        [int(s) for s in support.split()]))
        else:
            story[num] = rest.strip()
        # note: bAbI numbers questions too, so `story` deliberately skips them
    return out


TASKS = {1:  'qa1_single-supporting-fact',
         15: 'qa15_basic-deduction',
         16: 'qa16_basic-induction',
         19: 'qa19_path-finding'}
HOPS = {1: 1, 15: 2, 16: 3, 19: 2}      # supporting facts per question


def load_split(split):
    examples = []
    for task, name in TASKS.items():
        for ex in parse_babi(os.path.join(BABI_DIR, f'{name}_{split}.txt')):
            examples.append((task,) + ex)
    random.Random(SEED).shuffle(examples)
    return examples


train_examples, test_examples = load_split('train'), load_split('test')
print(f'{len(train_examples)} train questions, {len(test_examples)} test questions '
      f'(bAbI ships disjoint train/test files)\n')

for task in TASKS:
    story, q, a, sup = next(e[1:] for e in train_examples if e[0] == task)
    print(f'task {task} ({HOPS[task]} hop{"s" if HOPS[task] > 1 else ""}) -- {q}  ->  {a}')
    for i in sup:
        print(f'      supporting: {story[i]}')

40000 train questions, 4000 test questions (bAbI ships disjoint train/test files)

task 1 (1 hop) -- Where is John?  ->  office
      supporting: John went to the office.
task 15 (2 hops) -- What is winona afraid of?  ->  wolf
      supporting: Winona is a cat.
      supporting: Cats are afraid of wolves.
task 16 (3 hops) -- What color is Greg?  ->  white
      supporting: Greg is a lion.
      supporting: Lily is a lion.
      supporting: Lily is white.
task 19 (2 hops) -- How do you go from the office to the bathroom?  ->  w,w
      supporting: The hallway is west of the office.
      supporting: The bathroom is west of the hallway.


---
## 3. Two renderings of the same question

A word-level tokenizer here rather than the repo's character-level one. bAbI sentences are
English, and at the character level a single story is ~250 characters before the chain is
even added; word tokens make each example ~50 tokens and the whole vocabulary ~50 entries.
Nothing in `Transformer` cares — it is an embedding table either way — and
`DataLoader.tinyDataLoader` batches a list of integer ids happily, so the training loop is
unchanged.

Three control tokens: `=` ends the question and hands over to the model, `#` marks the
final answer, `</s>` ends the example.

In [5]:
def tokenize(text):
    return re.findall(r'[a-z]+|[.?,]', text.lower())


def render(story, question, answer, support, cot):
    '''Flatten one example into tokens. cot=True inserts the gold supporting facts.'''
    out = []
    for i in sorted(story):
        out += tokenize(story[i])
    out += tokenize(question) + ['=']
    if cot:
        for i in support:            # bAbI lists them in reasoning order, not story order
            out += tokenize(story[i])
    return out + ['#'] + tokenize(answer) + ['</s>']


cot_tokens    = [t for task, *e in train_examples for t in render(*e, cot=True)]
direct_tokens = [t for task, *e in train_examples for t in render(*e, cot=False)]

VOCAB = sorted(set(cot_tokens + direct_tokens))
stoi = {w: i for i, w in enumerate(VOCAB)}
EOS = stoi['</s>']

print('vocabulary', len(VOCAB), 'tokens:', VOCAB, '\n')
print('CoT   :', ' '.join(render(*train_examples[0][1:], cot=True)), '\n')
print('direct:', ' '.join(render(*train_examples[0][1:], cot=False)), '\n')
print('corpus: CoT', len(cot_tokens), 'tokens | direct', len(direct_tokens), 'tokens')
print('longest example', max(len(render(*e[1:], cot=True)) for e in train_examples), 'tokens')

vocabulary 68 tokens: ['#', ',', '.', '</s>', '=', '?', 'a', 'afraid', 'are', 'back', 'bathroom', 'bedroom', 'bernhard', 'brian', 'cat', 'cats', 'color', 'daniel', 'do', 'e', 'east', 'emily', 'frog', 'from', 'garden', 'gertrude', 'go', 'gray', 'green', 'greg', 'hallway', 'how', 'is', 'jessica', 'john', 'journeyed', 'julius', 'kitchen', 'lily', 'lion', 'mary', 'mice', 'mouse', 'moved', 'n', 'north', 'of', 'office', 'rhino', 's', 'sandra', 'sheep', 'south', 'swan', 'the', 'to', 'travelled', 'w', 'went', 'west', 'what', 'where', 'white', 'winona', 'wolf', 'wolves', 'yellow', 'you'] 

CoT   : john travelled to the bedroom . john journeyed to the bathroom . sandra moved to the hallway . daniel moved to the kitchen . john travelled to the bedroom . john went to the office . where is john ? = john went to the office . # office </s> 

direct: john travelled to the bedroom . john journeyed to the bathroom . sandra moved to the hallway . daniel moved to the kitchen . john travelled to the bedroo

longest example 83 tokens


In [6]:
def build_stream(cot, mask_prompt=False):
    '''Pack the corpus as an [N, 2] array: column 0 is token ids, column 1 is the loss
    weight. tinyDataLoader slices rows, so the weight rides along with its token and
    stays aligned after shuffling. mask_prompt=True trains only on the response, i.e.
    everything after '=' (used in section 10).'''
    ids, weight = [], []
    for task, story, q, a, sup in train_examples:
        toks = render(story, q, a, sup, cot)
        cut = toks.index('=')
        ids += [stoi[t] for t in toks]
        weight += ([0] * (cut + 1) + [1] * (len(toks) - cut - 1)) if mask_prompt \
                  else [1] * len(toks)
    return np.stack([np.array(ids), np.array(weight)], axis=1)


print(f'response tokens (everything after "="): '
      f'{build_stream(True, True)[:, 1].mean():.0%} of the CoT corpus, '
      f'{build_stream(False, True)[:, 1].mean():.0%} of the direct corpus')

response tokens (everything after "="): 24% of the CoT corpus, 7% of the direct corpus


That last number is worth noting before we start. Writing the reasoning down does not just
give the model room to compute at inference time — it also **turns one label into many**.
A direct example supervises essentially two tokens (the answer and the terminator); a CoT
example supervises a whole chain of sentences the model has to retrieve correctly. Denser
supervision is a second, less-discussed reason CoT training works, and it is a confound
worth keeping in mind when reading the results: the two conditions are matched on
optimizer steps and on architecture, not on the number of answer-bearing tokens.

### Config

The repo's `Configs`, with `vocab_size` matched to the new alphabet and `seq_length` set to
128 — comfortably longer than the longest example, so a training block usually holds a
whole story plus its chain.

In [7]:
def build_config():
    c = Training.Configs()
    c.device         = DEVICE
    c.vocab_size     = len(VOCAB)
    c.seq_length     = 128
    c.max_seq_length = 512
    c.batch_size     = 64
    c.d_model        = 128
    c.num_heads      = 8
    c.num_blocks     = 4
    c.num_groups     = 4
    c.attention_type = 'grouped_query_attention'
    c.ff             = 'mlp_with_gelu'
    c.use_experts    = False
    c.pos_embed      = 'rope'
    c.inference      = False      # we decode without the KV cache, see section 5
    return c


config = build_config()
print(f'{sum(p.numel() for p in TransformerMain.Transformer(config).parameters())/1e6:.2f}M '
      f'parameters per model')

0.92M parameters per model


---
## 4. Training

The loop from `Training.training()`: `DataLoader.tinyDataLoader` packs the token stream
into contiguous blocks and the loss is next-token cross entropy. Three changes — a cosine
learning-rate decay, a per-token loss weight (all ones for now; section 10 uses it), and a
budget counted in **optimizer steps** rather than epochs. Steps matter for fairness: the
CoT corpus is longer in tokens, so equal epochs would quietly hand it more gradient
updates.

In [8]:
def train_model(stream, config, max_steps, lr=1e-3, tag='', log_every=1000):
    '''stream is an [N, 2] array from build_stream: token ids and per-token loss weights.'''
    model = TransformerMain.Transformer(config).to(config.device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_steps)

    t0, step = time.perf_counter(), 0
    while step < max_steps:
        for xb, yb in DataLoader.tinyDataLoader(stream, config.batch_size, config.seq_length):
            x = torch.tensor(xb[:, :, 0]).to(config.device)
            y = torch.tensor(yb[:, :, 0]).long().to(config.device)
            w = torch.tensor(yb[:, :, 1]).float().to(config.device)
            logits, _ = model(x)
            per_token = F.cross_entropy(logits.view(-1, config.vocab_size), y.view(-1),
                                        reduction='none')
            loss = (per_token * w.view(-1)).sum() / w.sum().clamp(min=1)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            step += 1
            if step % log_every == 0:
                print(f'  {tag} step {step:5d}  loss {loss.item():.4f}  '
                      f'({time.perf_counter()-t0:.0f}s)')
            if step >= max_steps:
                break
    return model


MAX_STEPS = 14000 if DEVICE == 'cuda' else 2000
if DEVICE != 'cuda':
    print('No GPU detected: running a reduced step budget. Absolute accuracies will be '
          'lower than the numbers discussed below, but the CoT/direct gap still shows.\n')

print('training the chain-of-thought model')
cot_model = train_model(build_stream(cot=True), config, MAX_STEPS, tag='cot')
print('\ntraining the direct-answer model')
direct_model = train_model(build_stream(cot=False), config, MAX_STEPS, tag='direct')

training the chain-of-thought model


  cot step  1000  loss 0.5013  (45s)


  cot step  2000  loss 0.5048  (92s)


  cot step  3000  loss 0.4778  (140s)


  cot step  4000  loss 0.4774  (190s)


...

  direct step 10000  loss 0.4594  (479s)


  direct step 11000  loss 0.4430  (530s)


  direct step 12000  loss 0.4353  (580s)


  direct step 13000  loss 0.4314  (630s)


  direct step 14000  loss 0.4503  (678s)


The two losses are not comparable to each other — they are averages over different text.
Most of the CoT model's tokens are copies of sentences already in its context, which is
easy; the direct model's tokens are mostly story text plus a hard answer.

---
## 5. Batched sampling

`Generate.py` decodes one sequence greedily. Self-consistency needs many *sampled* chains,
so here is a batched sampler with temperature, stopping each row at its first `</s>`.

It runs the uncached path (`config.inference = False`), re-encoding the prefix each step.
Slower per token, but the sequences are short and it avoids the KV cache's fixed
`config.batch_size` preallocation, which is shared across calls.

In [9]:
@torch.no_grad()
def generate(model, prompts, config, max_new=48, temperature=0.0, seed=None):
    '''Batched decode from token-list prompts. temperature=0 is greedy.
    All prompts in one call must be the same length.'''
    model.eval()
    assert len(set(map(len, prompts))) == 1, 'batch equal-length prompts only'

    gen = torch.Generator(device=config.device)
    if seed is not None:
        gen.manual_seed(seed)

    x = torch.tensor([[stoi[t] for t in p] for p in prompts], device=config.device)
    done = torch.zeros(len(prompts), dtype=torch.bool, device=config.device)

    for _ in range(max_new):
        logits = model(x[:, -config.seq_length:])[0][:, -1, :]
        if temperature == 0.0:
            nxt = logits.argmax(-1, keepdim=True)
        else:
            nxt = torch.multinomial(torch.softmax(logits / temperature, -1), 1, generator=gen)
        nxt = torch.where(done[:, None], torch.full_like(nxt, EOS), nxt)  # freeze finished rows
        x = torch.cat([x, nxt], dim=-1)
        done |= (nxt.squeeze(1) == EOS)
        if done.all():
            break

    outs = []
    for row in x.tolist():
        completion = [VOCAB[i] for i in row][len(prompts[0]):]
        outs.append(completion[:completion.index('</s>')] if '</s>' in completion
                    else completion)
    return outs


def make_prompt(story, question, force_answer=False):
    '''Everything up to and including '='; force_answer also supplies the '#',
    which leaves the model no room to write a chain.'''
    out = []
    for i in sorted(story):
        out += tokenize(story[i])
    out += tokenize(question) + ['=']
    return out + ['#'] if force_answer else out


def answer_of(completion):
    return completion[completion.index('#') + 1:] if '#' in completion else None


def evaluate(model, examples, force_answer=False, n=1000, temperature=0.0, seed=0, keep=0):
    '''Per-task exact-match accuracy. Prompts are batched by length.'''
    by_length = collections.defaultdict(list)
    for task, story, q, a, sup in examples[:n]:
        by_length[len(make_prompt(story, q, force_answer))].append((task, story, q, a, sup))

    stats, samples = collections.defaultdict(lambda: [0, 0]), []
    for group in by_length.values():
        outs = generate(model, [make_prompt(s, q, force_answer) for _, s, q, _, _ in group],
                        config, temperature=temperature, seed=seed)
        for (task, story, q, a, sup), out in zip(group, outs):
            gold = tokenize(a)
            got = out[:len(gold)] if force_answer else answer_of(out)
            stats[task][0] += (got == gold)
            stats[task][1] += 1
            if len(samples) < keep:
                samples.append((task, q, out, gold))

    per_task = {t: c / n_ for t, (c, n_) in sorted(stats.items())}
    overall = sum(c for c, _ in stats.values()) / sum(n_ for _, n_ in stats.values())
    return per_task, overall, samples


def report(title, per_task, overall):
    cells = '  '.join(f'{t}({HOPS[t]}h): {a:5.1%}' for t, a in per_task.items())
    print(f'{title:<32} overall {overall:6.1%}   |   {cells}')

---
## 6. Does the chain of thought help?

Greedy decoding on 1000 held-out questions from the bAbI test files, which are disjoint
stories from training. `1(1h)` is task 1 at one hop, and so on.

In [10]:
cot_per, cot_all, cot_s = evaluate(cot_model, test_examples, keep=4)
dir_per, dir_all, dir_s = evaluate(direct_model, test_examples, keep=4)

report('chain of thought', cot_per, cot_all)
report('direct answer', dir_per, dir_all)

print('\nCoT model, sample completions:')
for task, q, out, gold in cot_s:
    mark = 'ok ' if answer_of(out) == gold else 'BAD'
    print(f'  {mark} [task {task}] {q}')
    print(f'      {" ".join(out)}   (true: {" ".join(gold)})')

print('\nper-task gain from the chain:')
print(f'  {"task":>6} {"hops":>5} {"direct":>8} {"CoT":>8} {"gain":>8}')
for t in cot_per:
    print(f'  {t:>6} {HOPS[t]:>5} {dir_per[t]:>8.1%} {cot_per[t]:>8.1%} '
          f'{cot_per[t]-dir_per[t]:>+8.1%}')

chain of thought                 overall  65.9%   |   1(1h): 98.8%  15(2h): 99.6%  16(3h): 44.9%  19(2h): 16.8%
direct answer                    overall  50.7%   |   1(1h): 96.9%  15(2h): 49.2%  16(3h): 48.6%  19(2h):  6.1%

CoT model, sample completions:
  ok  [task 16] What color is Julius?
      julius is a frog . bernhard is a frog . bernhard is gray . # gray   (true: gray)
  BAD [task 16] What color is Lily?
      lily is a frog . greg is a frog . greg is gray . # gray   (true: white)
  BAD [task 16] What color is Greg?
      greg is a swan . brian is a swan . brian is white . # white   (true: yellow)
  BAD [task 16] What color is Bernhard?
      bernhard is a swan . lily is a swan . lily is green . # green   (true: white)

per-task gain from the chain:
    task  hops   direct      CoT     gain
       1     1    96.9%    98.8%    +2.0%
      15     2    49.2%    99.6%   +50.4%
      16     3    48.6%    44.9%    -3.7%
      19     2     6.1%    16.8%   +10.7%


Read the per-task column, not the overall number — that is where the control does its work.

**Task 1, the control, behaves exactly as predicted: 98.8% vs 96.9%, a gain of +2.0%.**
A one-hop lookup needs no chain, and writing the sentence out first adds essentially
nothing. Had the gap appeared uniformly across all four tasks, the honest conclusion would
have been the boring one — that the CoT model is simply better trained. It doesn't.

**Task 15 is the clean win: 99.6% vs 49.2%, +50.4%.** Two hops, and the direct model sits
near the rate you get from guessing among the four possible answers while the CoT model
essentially solves it.

**Tasks 16 and 19 are where it gets interesting, and neither is a success story.** Task 19
gains +10.7% but from a terrible base (16.8% vs 6.1%), and task 16 — the task with the
*most* hops — actually comes out slightly worse with the chain, −3.7%. So the tidy
"gain scales with hops" story is wrong, and section 9 digs into why.

Two caveats on these numbers, both real:

- **Task 19 is unstable across seeds.** A separate run with an identical config gave the
  CoT model 52.9% on task 19 rather than 16.8%. It sits on a knife edge between learning
  the direction inversion and not; treat any single number for it as one draw, not a
  measurement. Tasks 1, 15 and 16 reproduced closely across my runs.
- **Both models were still improving at 14k steps.** These are equal-budget comparisons,
  not converged ones.

---
## 7. Ablation: make the CoT model skip its own chain

The sharpest test that the *tokens* are doing the work rather than the CoT model simply
being a better model. Same weights, but the prompt now ends `... ? = #`, so the `#` is
supplied by us and the model must answer immediately instead of retrieving first.

In [11]:
force_per, force_all, force_s = evaluate(cot_model, test_examples, force_answer=True, keep=3)

report('CoT model, free-running', cot_per, cot_all)
report('CoT model, chain skipped', force_per, force_all)
report('direct model', dir_per, dir_all)

print('\nsame weights, no room to retrieve:')
for task, q, out, gold in force_s:
    print(f'  [task {task}] {q} -> {" ".join(out[:4])}   (true: {" ".join(gold)})')

CoT model, free-running          overall  65.9%   |   1(1h): 98.8%  15(2h): 99.6%  16(3h): 44.9%  19(2h): 16.8%
CoT model, chain skipped         overall  13.7%   |   1(1h): 32.2%  15(2h):  4.3%  16(3h): 16.9%  19(2h):  1.2%
direct model                     overall  50.7%   |   1(1h): 96.9%  15(2h): 49.2%  16(3h): 48.6%  19(2h):  6.1%

same weights, no room to retrieve:
  [task 16] What color is Julius? -> gray   (true: gray)
  [task 16] What color is Lily? -> office   (true: white)
  [task 16] What color is Greg? -> yellow   (true: yellow)


Overall accuracy collapses from **65.9% to 13.7%** — far below the separately trained
direct model at 50.7%, and on task 15 from 99.6% to 4.3%. The weights that answer nearly
every task-15 question are the same weights failing here. What was removed is not
knowledge, it is the opportunity to write intermediate tokens.

(The forced condition also carries a format penalty: this model was never trained to place
an answer straight after `#`, so part of the drop is unfamiliarity with the output shape.
Read the direction and scale, not the precise value — and note the drop is far too large
to be format alone.)

---
## 8. Self-consistency

Sample `k` chains instead of taking one greedy chain, and answer with the majority vote
(Wang et al., 2022). Sampling errors are independent and scatter across many wrong
retrievals, while the correct chain is a single attractor that good samples agree on. This
only pays off where sampling actually introduces errors, so we sweep temperature.

In [12]:
def self_consistency(model, examples, k=8, temperature=1.0, n=150, seed=0):
    single = majority = 0
    for j, (task, story, q, a, sup) in enumerate(examples[:n]):
        gold = tokenize(a)
        outs = generate(model, [make_prompt(story, q)] * k, config,
                        temperature=temperature, seed=seed + j)
        answers = [tuple(x) for x in (answer_of(o) for o in outs) if x is not None]
        if not answers:
            continue
        single += (list(answers[0]) == gold)
        majority += (list(collections.Counter(answers).most_common(1)[0][0]) == gold)
    return single / n, majority / n


print(f'{"temperature":>12} {"1 chain":>10} {"majority of 8":>15} {"gain":>8}')
for T in [0.7, 1.0, 1.3, 1.6]:
    one, many = self_consistency(cot_model, test_examples, k=8, temperature=T)
    print(f'{T:>12} {one:>10.1%} {many:>15.1%} {many-one:>+8.1%}')

 temperature    1 chain   majority of 8     gain


         0.7      65.3%           66.0%    +0.7%


         1.0      66.7%           66.7%    +0.0%


         1.3      63.3%           67.3%    +4.0%


         1.6      60.7%           65.3%    +4.7%


The gains are modest here — a few points at high temperature — because greedy decoding is
already near this model's ceiling, and because most of its errors are not sampling noise.
Self-consistency fixes chains that go wrong *randomly*; the failures on tasks 16 and 19 are
systematic, and the model makes the same wrong retrieval on every sample. Voting eight
identical mistakes still gives you the mistake.

That is a useful negative result to have in hand: self-consistency buys robustness to
sampling noise, not capability, and when accuracy is limited by a systematic retrieval
failure it has almost nothing to work with.

---
## 9. Is the chain faithful?

This is what bAbI gives us that a homemade task does not: the dataset states which
sentences are the reasoning. So "faithfulness" is not a judgement call, it is a
comparison against annotation. Three things to check on the greedy chains:

1. **Retrieval accuracy** — does the emitted chain equal the gold supporting facts, in
   the gold order?
2. **Hallucination** — is every emitted sentence actually present in the story, or does
   the model invent plausible sentences?
3. **Load-bearing** — is the answer right when the retrieval is right, and wrong when it
   is not? If the two are independent, the chain is decoration.

In [13]:
def split_sentences(tokens):
    '''Chain tokens -> list of sentences (each a tuple of tokens, including the '.').'''
    sentences, current = [], []
    for t in tokens:
        current.append(t)
        if t == '.':
            sentences.append(tuple(current)); current = []
    if current:
        sentences.append(tuple(current))
    return sentences


rows = []
by_length = collections.defaultdict(list)
for task, story, q, a, sup in test_examples[:1000]:
    by_length[len(make_prompt(story, q))].append((task, story, q, a, sup))

for group in by_length.values():
    outs = generate(cot_model, [make_prompt(s, q) for _, s, q, _, _ in group], config)
    for (task, story, q, a, sup), out in zip(group, outs):
        chain = split_sentences(out[:out.index('#')] if '#' in out else out)
        gold_chain = [tuple(tokenize(story[i])) for i in sup]
        in_story = {tuple(tokenize(s)) for s in story.values()}
        rows.append(dict(task=task,
                         exact=chain == gold_chain,
                         as_set=set(chain) == set(gold_chain),
                         grounded=all(s in in_story for s in chain),
                         # was each hop a real story sentence? (position -> bool)
                         per_hop=[s in in_story for s in chain],
                         n_sent=len(chain),
                         correct=answer_of(out) == tokenize(a)))

def frac(rs, key):
    return sum(r[key] for r in rs) / len(rs) if rs else float('nan')

print(f'{"task":>6} {"hops":>5} {"chain exact":>12} {"right facts":>12} '
      f'{"grounded":>10} {"answer":>8}')
for t in sorted(TASKS):
    rs = [r for r in rows if r['task'] == t]
    print(f'{t:>6} {HOPS[t]:>5} {frac(rs,"exact"):>12.1%} {frac(rs,"as_set"):>12.1%} '
          f'{frac(rs,"grounded"):>10.1%} {frac(rs,"correct"):>8.1%}')

right, wrong = [r for r in rows if r['exact']], [r for r in rows if not r['exact']]
print(f'\nchains matching the gold supporting facts exactly: {len(right)}/{len(rows)}')
print(f'  answer accuracy | chain correct   : {frac(right, "correct"):.1%}')
print(f'  answer accuracy | chain wrong     : {frac(wrong, "correct"):.1%}')
print(f'  chains with an invented sentence  : {1 - frac(rows, "grounded"):.1%}')

# which hop does the model get wrong? a real story sentence at position i means the
# retrieval succeeded there; an invented one means it generated from the template instead
print('\ngrounding by position in the chain (is this sentence really in the story?)')
print(f'  {"task":>6}' + ''.join(f'{f"hop {i+1}":>10}' for i in range(3)))
for t in sorted(TASKS):
    rs = [r for r in rows if r['task'] == t]
    cells = ''
    for i in range(3):
        hits = [r['per_hop'][i] for r in rs if len(r['per_hop']) > i]
        cells += f'{sum(hits)/len(hits):>10.1%}' if hits else f'{"-":>10}'
    print(f'  {t:>6}' + cells)

  task  hops  chain exact  right facts   grounded   answer
     1     1        96.9%        96.9%      97.6%    98.8%
    15     2        99.6%        99.6%      99.6%    99.6%
    16     3        18.9%        18.9%      23.0%    44.9%
    19     2        16.4%        16.4%      16.4%    16.8%

chains matching the gold supporting facts exactly: 590/1000
  answer accuracy | chain correct   : 98.6%
  answer accuracy | chain wrong     : 18.8%
  chains with an invented sentence  : 39.8%

grounding by position in the chain (is this sentence really in the story?)
    task     hop 1     hop 2     hop 3
       1     97.6%         -         -
      15     99.6%    100.0%         -
      16     73.3%     38.3%     97.9%
      19     33.2%     33.7%         -


In [14]:
# What a broken chain actually looks like, on the two tasks where CoT struggles.
for task in (16, 19):
    print('=' * 78, f'\ntask {task}\n')
    group = [e for e in test_examples if e[0] == task][:3]
    outs = generate(cot_model, [make_prompt(s, q) for _, s, q, _, _ in group], config)
    for (t, story, q, a, sup), out in zip(group, outs):
        print(f'  {q}   (answer: {a})')
        print(f'    gold chain : ' + ' | '.join(story[i] for i in sup))
        print(f'    model chain: ' + ' '.join(out))
        print()

task 16

  What color is Julius?   (answer: gray)
    gold chain : Julius is a frog. | Brian is a frog. | Brian is gray.
    model chain: julius is a frog . bernhard is a frog . bernhard is gray . # gray

  What color is Lily?   (answer: white)
    gold chain : Lily is a frog. | Julius is a frog. | Julius is white.
    model chain: lily is a frog . greg is a frog . greg is gray . # gray

  What color is Greg?   (answer: yellow)
    gold chain : Greg is a lion. | Bernhard is a lion. | Bernhard is yellow.
    model chain: greg is a swan . brian is a swan . brian is white . # white

task 19

  How do you go from the bedroom to the kitchen?   (answer: s,e)
    gold chain : The bedroom is north of the garden. | The garden is west of the kitchen.
    model chain: the bedroom is north of the garden . the kitchen is north of the garden . # n , n

  How do you go from the hallway to the bathroom?   (answer: w,s)
    gold chain : The office is west of the hallway. | The office is north of the ba

This is the most informative output in the notebook. The chains are **structurally
perfect**: right number of sentences, right templates, right entity in the first slot, and
the answer faithfully copied from the last sentence. What is wrong is the *content* of the
middle hop — the model writes `greg is a swan` when the story says lion, or
`the kitchen is north of the garden` when the story says west. It is generating a plausible
sentence from the template instead of copying the one that is actually in the story.

So the failure on tasks 16 and 19 is **retrieval, not composition**. The grounding table
above localises it precisely: on task 16 the first hop is a real story sentence 73.3% of
the time and the second only **38.3%**.

And notice what separates these tasks from task 15, which the model nearly solves. In task
15 every supporting fact can be found using words from the question itself (`gertrude`).
In tasks 16 and 19 the second hop must be found using a key the model has just *derived* —
the type it retrieved in hop 1 (`swan`), or the intermediate room. Searching on a derived
key is the hard part, and it is exactly the step the annotated chain does not decompose any
further. The chain tells the model *what to write*; it does not break the search itself
into smaller pieces.

(The 97.9% at hop 3 on task 16 is not a recovery — it is the model staying consistent with
its own invention. Having decided `greg is a frog`, it then correctly copies `greg is
gray`, a real sentence. Grounded, and still wrong.)

The conditional accuracies are the payoff, and they are stark: **98.6% accurate when the
chain matches bAbI's supporting facts, 18.8% when it does not.** Retrieval and correctness
are not independent variables that happen to correlate — the chain is the causal path to
the answer, which is why a wrong answer arrives with a localisable cause attached.

This is the property that makes CoT worth more than its accuracy gain. On task 16 the model
is right 44.9% of the time, but we are not left guessing why it fails: we can point at hop
2 and say the search on the derived key is what broke. A direct model gives you a wrong
token and nothing else.

---
## 10. Summary

| | overall | task 1 (1 hop) | task 15 (2 hop) | task 16 (3 hop) | task 19 (2 hop) |
| --- | --- | --- | --- | --- | --- |
| direct answer | 50.7% | 96.9% | 49.2% | 48.6% | 6.1% |
| CoT, chain skipped (ablation) | 13.7% | 32.2% | 4.3% | 16.9% | 1.2% |
| **chain of thought** | **65.9%** | **98.8%** | **99.6%** | 44.9% | 16.8% |

What the experiment supports:

1. **The chain does the work, not the weights.** Same model, forced past its own chain,
   drops from 65.9% to 13.7%.
2. **The benefit is specific to multi-hop questions.** The one-hop control gains +2.0%;
   task 15 gains +50.4%.
3. **The chain is faithful and therefore diagnostic.** 98.6% accuracy when the retrieved
   facts match bAbI's annotation versus 18.8% when they don't, and the answer is copied
   from the last chain sentence.

What it does not support, and where I would not push the claim:

- **More hops does not mean more gain.** Task 16 has the most hops and gains nothing
  (−3.7%). Hop count is the wrong variable; whether the next fact can be found from the
  question's own words, rather than from a key the model derived, predicts these results
  much better.
- **Task 19's number is one draw**, not a measurement (16.8% here, 52.9% on a rerun).
- **Neither model is converged**, and the CoT format supervises ~3× more response tokens
  per question, so the comparison is matched on optimizer steps and architecture but not
  on supervision density.

In [15]:
print(f'{"":<36}{"overall":>9}' + ''.join(f'{f"task {t}":>10}' for t in sorted(TASKS)))
print(f'{"":<36}{"":>9}' + ''.join(f'{f"({HOPS[t]} hop)":>10}' for t in sorted(TASKS)))
for name, per, overall in [('direct answer', dir_per, dir_all),
                           ('CoT, chain skipped (ablation)', force_per, force_all),
                           ('chain of thought', cot_per, cot_all)]:
    print(f'{name:<36}{overall:>9.1%}' + ''.join(f'{per[t]:>10.1%}' for t in sorted(TASKS)))

                                      overall    task 1   task 15   task 16   task 19
                                                (1 hop)   (2 hop)   (3 hop)   (2 hop)
direct answer                           50.7%     96.9%     49.2%     48.6%      6.1%
CoT, chain skipped (ablation)           13.7%     32.2%      4.3%     16.9%      1.2%
chain of thought                        65.9%     98.8%     99.6%     44.9%     16.8%


In [16]:
SAVE = False   # flip to True to write these models into SavedModels/

if SAVE:
    from dataclasses import asdict
    for name, model in [('model_babi_cot.pth', cot_model),
                        ('model_babi_direct.pth', direct_model)]:
        path = os.path.join('SavedModels', name)
        # TransformerMain.saveModel's layout plus the tokenizer vocabulary, which these
        # models need and which is not TinyShakespeare's alphabet
        torch.save({'state_dict': model.state_dict(),
                    'config': asdict(model.config),
                    'vocab': VOCAB}, path)
        print('saved', path)
else:
    print('SAVE is False - nothing written to SavedModels/')

SAVE is False - nothing written to SavedModels/


### Where to take it next

- **More hops.** Tasks 2 and 3 (two and three supporting facts over a long, distractor-filled
  story) are the real test of retrieval under interference. They need a larger `seq_length`
  — stories run to ~350 words — but `parse_babi` already handles them: add them to `TASKS`.
- **Attention variants.** Swap `attention_type` in `build_config()`. Sliding-window
  attention should hurt specifically when a supporting fact falls outside the window, which
  is a targeted prediction this setup can check per-task.
- **Mixture of experts.** Turn on `use_experts` and ask whether the router specialises by
  chain position — retrieval tokens versus answer tokens.
- **Response-only loss.** `build_stream(mask_prompt=True)` zeroes the loss weight on
  everything before `=`, so the gradient lands only on the chain and the answer. In my
  trials this helped the CoT model on task 19 but destabilised the direct baseline — with
  ~2 supervised tokens per example it has very little left to learn from — so it needs a
  larger step budget for both arms before the comparison means anything. Worth doing
  properly.
- **A finer-grained chain for task 19.** The annotated chain gives the two room facts but
  still asks the model to invert them into directions in a single step. Rendering the
  derived directions as their own chain steps would test whether the remaining failure is
  the search or the inversion.
- **Wrong chains on purpose.** Feed the model a chain with one supporting fact corrupted
  and see whether the answer follows the corrupted chain. If it does, the chain is causal;
  if the model "recovers", it was not really using it.